# Model B v2 — GPA Prediction (Fresh Build)

**Key improvements over v1:**
1. **Z-score cleaning on features only** — preserves all GPA values
2. **Smart feature engineering** — interaction terms with study_hours × stress_level
3. **5 diverse algorithms** compared (LightGBM, XGBoost, RF, Ridge, SVR)
4. **Optuna tuning** with 10-fold CV for stable estimates on small data
5. **Feature importance pruning** — removes noise for 2K-row dataset
6. **Stacking ensemble** with diverse base models

In [7]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, KFold, RepeatedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, StackingRegressor

import lightgbm as lgb
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("All imports OK.")

All imports OK.


## 1. Load & Clean Data

In [8]:
df = pd.read_csv("../../Datasets/student_lifestyle_dataset.csv")
df = df.drop(columns=["Student_ID"])
df = df.rename(columns={
    "Study_Hours_Per_Day": "study_hours",
    "Extracurricular_Hours_Per_Day": "eca_hours",
    "Sleep_Hours_Per_Day": "sleep_hours",
    "Social_Hours_Per_Day": "social_hours",
    "Physical_Activity_Hours_Per_Day": "physical_hours",
    "Stress_Level": "stress_level",
    "GPA": "gpa",
})

print(f"Raw: {len(df)} rows")

Raw: 2000 rows


In [9]:
# Encode stress level
stress_mapping = {"Low": 0, "Moderate": 1, "High": 2}
df["stress_level"] = df["stress_level"].map(stress_mapping)

# Z-score cleaning on FEATURES only (not GPA)
feature_cols_raw = ["study_hours", "eca_hours", "sleep_hours", "social_hours", "physical_hours", "stress_level"]
z = np.abs(stats.zscore(df[feature_cols_raw].select_dtypes("number")))
mask = (z < 3).all(axis=1)
df = df[mask]

print(f"After Z-score cleaning (features only): {len(df)} rows")
print(f"GPA range preserved: {df['gpa'].min():.2f} – {df['gpa'].max():.2f}")

After Z-score cleaning (features only): 1996 rows
GPA range preserved: 2.24 – 4.00


## 2. Feature Engineering

In [10]:
def engineer_features_b(df):
    """Feature engineering for Model B. Applied to both train and inference."""
    df = df.copy()

    # Key interaction: the two strongest predictors combined
    df["study_x_stress"]      = df["study_hours"] * df["stress_level"]

    # Ratios capturing lifestyle balance
    df["study_leisure_ratio"]  = df["study_hours"] / (df["social_hours"] + df["physical_hours"] + 1)
    df["study_proportion"]     = df["study_hours"] / (24 - df["sleep_hours"] + 1e-6)

    # Other meaningful interactions
    df["physical_x_stress"]    = df["physical_hours"] * df["stress_level"]
    df["sleep_x_stress"]       = df["sleep_hours"] * df["stress_level"]

    # Non-linear terms for strongest predictor
    df["study_hours_sq"]       = df["study_hours"] ** 2

    # Composite lifestyle indices
    df["academic_intensity"]   = df["study_hours"] + df["eca_hours"] - df["social_hours"]
    df["total_committed"]      = df["study_hours"] + df["eca_hours"] + df["physical_hours"]
    df["wellbeing_index"]      = df["sleep_hours"] + df["social_hours"] - df["stress_level"]

    return df

df = engineer_features_b(df)
all_features = [c for c in df.columns if c != "gpa"]
print(f"Total features: {len(all_features)}")
print(f"Features: {all_features}")

Total features: 15
Features: ['study_hours', 'eca_hours', 'sleep_hours', 'social_hours', 'physical_hours', 'stress_level', 'study_x_stress', 'study_leisure_ratio', 'study_proportion', 'physical_x_stress', 'sleep_x_stress', 'study_hours_sq', 'academic_intensity', 'total_committed', 'wellbeing_index']


## 3. Train/Test Split

In [11]:
X = df[all_features]
y = df["gpa"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

Train: 1596 | Test: 400


## 4. Baseline Comparison

In [12]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

baselines = {
    "LightGBM":     lgb.LGBMRegressor(n_estimators=500, random_state=42, verbosity=-1),
    "XGBoost":      xgb.XGBRegressor(n_estimators=500, random_state=42, verbosity=0),
    "RandomForest": RandomForestRegressor(n_estimators=500, random_state=42),
    "Ridge":        Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=1.0))]),
    "SVR":          Pipeline([("scaler", StandardScaler()), ("svr", SVR(kernel="rbf"))]),
}

print("10-Fold CV Baseline:")
print("=" * 55)
for name, model in baselines.items():
    scores = cross_val_score(model, X_train, y_train, cv=kf, scoring="r2")
    print(f"  {name:15s} | R² = {scores.mean():.4f} ± {scores.std():.4f}")

10-Fold CV Baseline:
  LightGBM        | R² = 0.3897 ± 0.0630
  XGBoost         | R² = 0.3768 ± 0.0510
  RandomForest    | R² = 0.4824 ± 0.0281
  Ridge           | R² = 0.5350 ± 0.0281
  SVR             | R² = 0.5005 ± 0.0318


## 5. Optuna Tuning

10-fold CV inside Optuna for stable estimates. Heavy regularization for small dataset.

In [13]:
kf_tune = KFold(n_splits=10, shuffle=True, random_state=42)

def objective_lgbm(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 200, 2000),
        "max_depth":         trial.suggest_int("max_depth", 2, 6),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 7, 63),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 0.01, 50.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 0.01, 50.0, log=True),
        "verbosity": -1, "random_state": 42,
    }
    model = lgb.LGBMRegressor(**params)
    scores = cross_val_score(model, X_train, y_train, cv=kf_tune, scoring="r2")
    return scores.mean()

print("Tuning LightGBM...")
study_lgbm = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, n_trials=100, show_progress_bar=True)
print(f"Best LightGBM R²: {study_lgbm.best_value:.4f}")

Tuning LightGBM...


Best trial: 95. Best value: 0.529041: 100%|██████████| 100/100 [01:29<00:00,  1.12it/s]

Best LightGBM R²: 0.5290


In [14]:
def objective_xgb(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 200, 2000),
        "max_depth":         trial.suggest_int("max_depth", 2, 6),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "min_child_weight":  trial.suggest_int("min_child_weight", 5, 100),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 0.01, 50.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 0.01, 50.0, log=True),
        "verbosity": 0, "random_state": 42,
    }
    model = xgb.XGBRegressor(**params)
    scores = cross_val_score(model, X_train, y_train, cv=kf_tune, scoring="r2")
    return scores.mean()

print("Tuning XGBoost...")
study_xgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=100, show_progress_bar=True)
print(f"Best XGBoost R²: {study_xgb.best_value:.4f}")

Tuning XGBoost...


Best trial: 74. Best value: 0.531943: 100%|██████████| 100/100 [03:42<00:00,  2.22s/it]

Best XGBoost R²: 0.5319


In [15]:
def objective_rf(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 200, 1500),
        "max_depth":         trial.suggest_int("max_depth", 3, 12),
        "min_samples_split": trial.suggest_int("min_samples_split", 5, 50),
        "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 3, 50),
        "max_features":      trial.suggest_float("max_features", 0.3, 1.0),
        "random_state": 42,
    }
    model = RandomForestRegressor(**params)
    scores = cross_val_score(model, X_train, y_train, cv=kf_tune, scoring="r2")
    return scores.mean()

print("Tuning RandomForest...")
study_rf = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(objective_rf, n_trials=80, show_progress_bar=True)
print(f"Best RandomForest R²: {study_rf.best_value:.4f}")

Tuning RandomForest...


Best trial: 67. Best value: 0.531355: 100%|██████████| 80/80 [24:59<00:00, 18.75s/it]

Best RandomForest R²: 0.5314


In [16]:
def objective_ridge(trial):
    alpha = trial.suggest_float("alpha", 1e-3, 100.0, log=True)
    model = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=alpha))])
    scores = cross_val_score(model, X_train, y_train, cv=kf_tune, scoring="r2")
    return scores.mean()

def objective_svr(trial):
    params = {
        "C":       trial.suggest_float("C", 0.1, 100.0, log=True),
        "epsilon": trial.suggest_float("epsilon", 0.001, 0.5, log=True),
        "gamma":   trial.suggest_categorical("gamma", ["scale", "auto"]),
    }
    model = Pipeline([("scaler", StandardScaler()), ("svr", SVR(kernel="rbf", **params))])
    scores = cross_val_score(model, X_train, y_train, cv=kf_tune, scoring="r2")
    return scores.mean()

print("Tuning Ridge...")
study_ridge = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_ridge.optimize(objective_ridge, n_trials=50, show_progress_bar=True)
print(f"Best Ridge R²: {study_ridge.best_value:.4f}")

print("\nTuning SVR...")
study_svr = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_svr.optimize(objective_svr, n_trials=50, show_progress_bar=True)
print(f"Best SVR R²: {study_svr.best_value:.4f}")

Tuning Ridge...


Best trial: 21. Best value: 0.535112: 100%|██████████| 50/50 [00:02<00:00, 20.31it/s]


Best Ridge R²: 0.5351

Tuning SVR...


Best trial: 28. Best value: 0.520439: 100%|██████████| 50/50 [00:48<00:00,  1.03it/s]

Best SVR R²: 0.5204


## 6. Train Tuned Models & Feature Pruning

In [17]:
best_lgbm  = lgb.LGBMRegressor(**study_lgbm.best_params, random_state=42, verbosity=-1)
best_xgb   = xgb.XGBRegressor(**study_xgb.best_params, random_state=42, verbosity=0)
best_rf    = RandomForestRegressor(**study_rf.best_params, random_state=42)
best_ridge = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=study_ridge.best_params["alpha"]))])
svr_p = {k: v for k, v in study_svr.best_params.items()}
best_svr   = Pipeline([("scaler", StandardScaler()), ("svr", SVR(kernel="rbf", **svr_p))])

print("Tuned model test performance:")
print("=" * 60)
tuned = {}
for name, m in [("LightGBM", best_lgbm), ("XGBoost", best_xgb), ("RF", best_rf), ("Ridge", best_ridge), ("SVR", best_svr)]:
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    r2 = r2_score(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    tuned[name] = {"model": m, "R2": r2, "RMSE": rmse, "MAE": mae}
    print(f"  {name:12s} | R² = {r2:.4f} | RMSE = {rmse:.4f} | MAE = {mae:.4f}")

Tuned model test performance:
  LightGBM     | R² = 0.5155 | RMSE = 0.2056 | MAE = 0.1675
  XGBoost      | R² = 0.5196 | RMSE = 0.2048 | MAE = 0.1668
  RF           | R² = 0.5153 | RMSE = 0.2057 | MAE = 0.1675
  Ridge        | R² = 0.5211 | RMSE = 0.2044 | MAE = 0.1668
  SVR          | R² = 0.5187 | RMSE = 0.2050 | MAE = 0.1671


In [18]:
# Feature importance from LightGBM
best_lgbm.fit(X_train, y_train)
imp = pd.Series(best_lgbm.feature_importances_, index=all_features)
imp_pct = (imp / imp.sum() * 100).sort_values(ascending=False)

print("Feature importance (%):")
for feat, pct in imp_pct.items():
    bar = "#" * int(pct)
    print(f"  {feat:25s} : {pct:5.1f}%  {bar}")

# Keep features with >= 1% importance
keep_features = imp_pct[imp_pct >= 1].index.tolist()
drop_features = imp_pct[imp_pct < 1].index.tolist()
print(f"\nKeeping {len(keep_features)} features, dropping {len(drop_features)}: {drop_features}")

Feature importance (%):
  study_hours               :  26.4%  ##########################
  study_proportion          :  24.5%  ########################
  study_x_stress            :  12.6%  ############
  study_leisure_ratio       :   8.4%  ########
  sleep_hours               :   7.8%  #######
  study_hours_sq            :   6.1%  ######
  physical_x_stress         :   4.6%  ####
  wellbeing_index           :   2.8%  ##
  academic_intensity        :   2.2%  ##
  total_committed           :   1.2%  #
  physical_hours            :   1.1%  #
  sleep_x_stress            :   1.0%  
  social_hours              :   0.8%  
  eca_hours                 :   0.6%  
  stress_level              :   0.0%  

Keeping 11 features, dropping 4: ['sleep_x_stress', 'social_hours', 'eca_hours', 'stress_level']


In [19]:
# Re-evaluate with pruned features
X_train_p = X_train[keep_features]
X_test_p  = X_test[keep_features]

print("Pruned feature performance:")
print("=" * 60)
pruned = {}
for name, m_fn in [
    ("LightGBM", lambda: lgb.LGBMRegressor(**study_lgbm.best_params, random_state=42, verbosity=-1)),
    ("XGBoost",  lambda: xgb.XGBRegressor(**study_xgb.best_params, random_state=42, verbosity=0)),
    ("RF",       lambda: RandomForestRegressor(**study_rf.best_params, random_state=42)),
    ("Ridge",    lambda: Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=study_ridge.best_params["alpha"]))])),
    ("SVR",      lambda: Pipeline([("scaler", StandardScaler()), ("svr", SVR(kernel="rbf", **svr_p))])),
]:
    m = m_fn()
    m.fit(X_train_p, y_train)
    pred = m.predict(X_test_p)
    r2 = r2_score(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    pruned[name] = {"model": m, "R2": r2, "RMSE": rmse, "MAE": mae}
    print(f"  {name:12s} | R² = {r2:.4f} | RMSE = {rmse:.4f} | MAE = {mae:.4f}")

# Decide whether pruning helped
avg_full = np.mean([v["R2"] for v in tuned.values()])
avg_pruned = np.mean([v["R2"] for v in pruned.values()])
use_pruned = avg_pruned >= avg_full - 0.005
final_features = keep_features if use_pruned else all_features
final_models = pruned if use_pruned else tuned
print(f"\nUsing {'pruned' if use_pruned else 'full'} features ({len(final_features)} features)")

Pruned feature performance:
  LightGBM     | R² = 0.5160 | RMSE = 0.2055 | MAE = 0.1676
  XGBoost      | R² = 0.5194 | RMSE = 0.2048 | MAE = 0.1669
  RF           | R² = 0.5157 | RMSE = 0.2056 | MAE = 0.1674
  Ridge        | R² = 0.5197 | RMSE = 0.2047 | MAE = 0.1672
  SVR          | R² = 0.5195 | RMSE = 0.2048 | MAE = 0.1673

Using pruned features (11 features)


## 7. Stacking Ensemble

In [20]:
X_train_f = X_train[final_features]
X_test_f  = X_test[final_features]
X_full_f  = X[final_features]

stacking = StackingRegressor(
    estimators=[
        ("lgbm",  lgb.LGBMRegressor(**study_lgbm.best_params, random_state=42, verbosity=-1)),
        ("xgb",   xgb.XGBRegressor(**study_xgb.best_params, random_state=42, verbosity=0)),
        ("ridge", Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=study_ridge.best_params["alpha"]))])),
        ("svr",   Pipeline([("scaler", StandardScaler()), ("svr", SVR(kernel="rbf", **svr_p))])),
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=10,
    n_jobs=-1,
)

print("Fitting stacking ensemble...")
stacking.fit(X_train_f, y_train)
pred_s = stacking.predict(X_test_f)
r2_s = r2_score(y_test, pred_s)
rmse_s = np.sqrt(mean_squared_error(y_test, pred_s))
mae_s = mean_absolute_error(y_test, pred_s)
print(f"  Stacking    | R² = {r2_s:.4f} | RMSE = {rmse_s:.4f} | MAE = {mae_s:.4f}")

Fitting stacking ensemble...
  Stacking    | R² = 0.5211 | RMSE = 0.2044 | MAE = 0.1670


## 8. Final Comparison & CV

In [21]:
all_results = {**{k: {m: v[m] for m in ["R2","RMSE","MAE"]} for k, v in final_models.items()},
               "Stacking": {"R2": r2_s, "RMSE": rmse_s, "MAE": mae_s}}
results_df = pd.DataFrame(all_results).T.sort_values("R2", ascending=False)
print("\n=== FINAL COMPARISON ===")
print(results_df.to_string())
print(f"\nOld Model B: R² = 0.5333, RMSE = 0.2029, MAE = 0.1646")

best_name = results_df.index[0]
print(f"\n>>> Best model: {best_name}")


=== FINAL COMPARISON ===
                R2      RMSE       MAE
Stacking  0.521098  0.204438  0.166985
Ridge     0.519702  0.204736  0.167180
SVR       0.519458  0.204788  0.167268
XGBoost   0.519436  0.204792  0.166882
LightGBM  0.515981  0.205527  0.167586
RF        0.515669  0.205594  0.167413

Old Model B: R² = 0.5333, RMSE = 0.2029, MAE = 0.1646

>>> Best model: Stacking


In [22]:
best_final = stacking if best_name == "Stacking" else final_models[best_name]["model"]

rkf = RepeatedKFold(n_splits=10, n_repeats=3, random_state=42)
cv_r2   = cross_val_score(best_final, X_full_f, y, cv=rkf, scoring="r2")
cv_mae  = cross_val_score(best_final, X_full_f, y, cv=rkf, scoring="neg_mean_absolute_error")
cv_rmse = cross_val_score(best_final, X_full_f, y, cv=rkf, scoring="neg_root_mean_squared_error")

print(f"RepeatedKFold (10×3) of {best_name}:")
print(f"  CV R²   : {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"  CV MAE  : {(-cv_mae).mean():.4f} ± {(-cv_mae).std():.4f}")
print(f"  CV RMSE : {(-cv_rmse).mean():.4f} ± {(-cv_rmse).std():.4f}")

KeyboardInterrupt: 

## 9. Export

In [ ]:
import os
os.makedirs("../models", exist_ok=True)

final_export = stacking if best_name == "Stacking" else final_models[best_name]["model"]

joblib.dump(final_export, "../models/modelB.pkl")
print(f"Saved {best_name} to ../models/modelB.pkl")

joblib.dump(stress_mapping, "../models/stress_level_encoder_modelB.pkl")
print("Saved stress mapping.")

pipeline_b = {
    "engineer_fn": engineer_features_b,
    "stress_mapping": stress_mapping,
    "important_features": final_features,
}
joblib.dump(pipeline_b, "../models/pipeline_b.pkl")
print(f"Saved pipeline ({len(final_features)} features) to ../models/pipeline_b.pkl")

In [ ]:
# Verification
loaded_model = joblib.load("../models/modelB.pkl")
loaded_pipe  = joblib.load("../models/pipeline_b.pkl")

test_input = pd.DataFrame([{
    "study_hours": 7.5, "eca_hours": 2.0, "sleep_hours": 7.0,
    "social_hours": 3.0, "physical_hours": 4.0, "stress_level": 1,
}])

test_eng = loaded_pipe["engineer_fn"](test_input)
test_final = test_eng[loaded_pipe["important_features"]]
pred = loaded_model.predict(test_final)[0]
print(f"Test prediction: {pred:.2f}")
print(f"Clamped: {max(0.0, min(4.0, round(pred, 2))):.2f}")
print("\nVerification PASSED.")